<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Bitcoin Foundations
## Chapter 5 – Blocks, Merkle Trees, and Chain Structure

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


This first lab builds a Merkle root from a handful of toy transaction IDs. It mirrors the book’s explanation that blocks summarise many transactions into a single root hash, which is what actually gets committed into the header and chained over time.

In [1]:
import hashlib  # hashing functions
from typing import List  # type hints


def merkle_root(leaves: List[bytes]) -> bytes:
    level = leaves[:]  # copy initial leaves
    if not level:
        return b""  # empty tree case
    while len(level) > 1:  # iterate until one root remains
        if len(level) % 2 == 1:  # if odd number of nodes
            level.append(level[-1])  # duplicate last leaf (Bitcoin-style)
        new_level = []  # next level up
        for i in range(0, len(level), 2):  # pairwise combine
            h = hashlib.sha256(level[i] + level[i + 1]).digest()  # hash pair
            new_level.append(h)  # append parent hash
        level = new_level  # move up one level
    return level[0]  # single root hash


txids = [  # toy transaction IDs as bytes
    b"tx1",
    b"tx2",
    b"tx3",
    b"tx4",
]

root = merkle_root(txids)  # compute Merkle root
print("Merkle root (hex):", root.hex())  # show root as hex string


Merkle root (hex): 6991caa77e3d6dd3514fffdbfcf93e4a10ed129717746c4bb3454bb8bc81274a


You then represent a tiny chain of block headers and verify that each header’s `prev_hash` field matches the previous block’s `hash`. This captures the core continuity check nodes perform when following the best chain.

In [2]:
from dataclasses import dataclass  # structured data for headers
from typing import Optional  # optional types


@dataclass
class Header:  # minimal header model
    height: int  # block height
    hash: str  # block hash (placeholder)
    prev_hash: Optional[str]  # previous block hash


chain = [  # small chain of three headers
    Header(height=0, hash="h0", prev_hash=None),
    Header(height=1, hash="h1", prev_hash="h0"),
    Header(height=2, hash="h2", prev_hash="h1"),
]  # end of chain


def is_chain_valid(headers) -> bool:  # continuity check
    for i in range(1, len(headers)):  # check links from second onward
        if headers[i].prev_hash != headers[i - 1].hash:  # mismatch?
            return False  # invalid link
    return True  # all links consistent


print("Chain valid?", is_chain_valid(chain))  # expect True


Chain valid? True


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
